### System & GPU Information

In [1]:
import os
import sys
import shutil
import subprocess
import platform

print("=" * 70)
print("Notebook 3 — GEMM LLVM IR Optimization")
print("=" * 70)

print("Python:", sys.version.split()[0])
print("OS:", platform.platform())

print("\n=== GPU Information ===")
subprocess.run(["nvidia-smi"], check=False)

Notebook 3 — GEMM LLVM IR Optimization
Python: 3.13.16
OS: Linux-6.6.159+-x86_64-with-glibc2.39

=== GPU Information ===


CompletedProcess(args=['nvidia-smi'], returncode=0)

### Install LLVM / Clang Toolchain

In [2]:
import subprocess

print("=" * 70)
print("Installing LLVM 20 Toolchain")
print("=" * 70)

subprocess.run(
    ["apt-get", "update", "-qq"],
    check=True
)

subprocess.run(
    [
        "apt-get", "install", "-y",
        "llvm-20",
        "llvm-20-tools",
        "llvm-20-dev",
        "clang-20",
        "libmlir-20-dev",
        "mlir-20-tools"
    ],
    check=True
)

print("\nLLVM Installation Complete")

Installing LLVM 20 Toolchain

LLVM Installation Complete


### Setup LLVM Tool Paths

In [3]:
import shutil
import subprocess

print("=" * 70)
print("LLVM / Clang Toolchain Verification")
print("=" * 70)

llvm_tools = [
    "clang-20",
    "clang++-20",
    "opt-20",
    "llc-20",
    "llvm-as-20",
    "llvm-dis-20",
    "llvm-config-20",
    "mlir-opt-20",
]

for tool in llvm_tools:

    path = shutil.which(tool)

    if path is None:
        print(f"{tool:20s}: NOT FOUND")
        continue

    print(f"{tool:20s}: {path}")

print("\n=== LLVM Version ===")

if shutil.which("opt-20"):
    subprocess.run(["opt-20", "--version"], check=True)

LLVM / Clang Toolchain Verification
clang-20            : /usr/bin/clang-20
clang++-20          : /usr/bin/clang++-20
opt-20              : /usr/bin/opt-20
llc-20              : /usr/bin/llc-20
llvm-as-20          : /usr/bin/llvm-as-20
llvm-dis-20         : /usr/bin/llvm-dis-20
llvm-config-20      : /usr/bin/llvm-config-20
mlir-opt-20         : /usr/bin/mlir-opt-20

=== LLVM Version ===


### PyTorch & Triton Environment

In [4]:
import torch
import triton
import triton.language as tl

print("=" * 70)
print("PyTorch / Triton Environment")
print("=" * 70)

print("PyTorch version:", torch.__version__)
print("Triton version :", triton.__version__)
print("CUDA version   :", torch.version.cuda)

assert torch.cuda.is_available(), "CUDA GPU is not available!"

device = torch.cuda.get_device_properties(0)

print("\n=== GPU ===")
print("GPU Name:", device.name)
print("Compute Capability:", f"{device.major}.{device.minor}")
print("GPU Memory:", round(device.total_memory / 1024**3, 2), "GB")

print("\nEnvironment Check: PASS")

PyTorch / Triton Environment
PyTorch version: 2.11.0+cu130
Triton version : 3.6.0
CUDA version   : 13.0

=== GPU ===
GPU Name: Tesla T4
Compute Capability: 7.5
GPU Memory: 14.56 GB

Environment Check: PASS


### Create Project Directories

In [5]:
from pathlib import Path

PROJECT_ROOT = Path("/content/gemm_llvm_optimization")

DIRS = {
    "llvm_ir": PROJECT_ROOT / "llvm_ir",
    "optimized_ir": PROJECT_ROOT / "optimized_ir",
    "cfg": PROJECT_ROOT / "cfg",
    "passes": PROJECT_ROOT / "passes",
    "results": PROJECT_ROOT / "results",
}

for name, path in DIRS.items():
    path.mkdir(parents=True, exist_ok=True)
    print(f"{name:15s}: {path}")

print("\nProject directories created.")

llvm_ir        : /content/gemm_llvm_optimization/llvm_ir
optimized_ir   : /content/gemm_llvm_optimization/optimized_ir
cfg            : /content/gemm_llvm_optimization/cfg
passes         : /content/gemm_llvm_optimization/passes
results        : /content/gemm_llvm_optimization/results

Project directories created.


### Final Environment Verification

In [6]:
print("=" * 70)
print("Notebook 3 — Final Setup Verification")
print("=" * 70)

checks = {
    "PyTorch": torch.__version__,
    "Triton": triton.__version__,
    "CUDA Available": torch.cuda.is_available(),
    "GPU": torch.cuda.get_device_name(0),
    "Clang": shutil.which("clang-20"),
    "LLVM opt": shutil.which("opt-20"),
    "LLVM llc": shutil.which("llc-20"),
    "LLVM assembler": shutil.which("llvm-as-20"),
    "LLVM disassembler": shutil.which("llvm-dis-20"),
    "MLIR opt": shutil.which("mlir-opt-20"),
}

for name, value in checks.items():
    print(f"{name:22s}: {value}")

required_tools = [
    "clang-20",
    "opt-20",
    "llc-20",
    "llvm-as-20",
    "llvm-dis-20"
]

missing = [
    tool for tool in required_tools
    if shutil.which(tool) is None
]

print("\n" + "=" * 70)

if not missing and torch.cuda.is_available():
    print("SETUP STATUS: PASS")
else:
    print("SETUP STATUS: INCOMPLETE")
    print("Missing tools:", missing)

Notebook 3 — Final Setup Verification
PyTorch               : 2.11.0+cu130
Triton                : 3.6.0
CUDA Available        : True
GPU                   : Tesla T4
Clang                 : /usr/bin/clang-20
LLVM opt              : /usr/bin/opt-20
LLVM llc              : /usr/bin/llc-20
LLVM assembler        : /usr/bin/llvm-as-20
LLVM disassembler     : /usr/bin/llvm-dis-20
MLIR opt              : /usr/bin/mlir-opt-20

SETUP STATUS: PASS


### Step 3.1 — GEMM LLVM IR Extraction

In [7]:
import torch
import triton
import triton.language as tl

@triton.jit
def gemm_kernel(
    A, B, C,
    M: tl.constexpr,
    N: tl.constexpr,
    K: tl.constexpr,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    acc = tl.full(
        (BLOCK_M, BLOCK_N),
        0,
        tl.float32
    )

    for k in range(triton.cdiv(K, BLOCK_K)):

        a_ptrs = A + (
            offs_m[:, None] * K
            + (k * BLOCK_K + offs_k[None, :])
        )

        b_ptrs = B + (
            (k * BLOCK_K + offs_k[:, None]) * N
            + offs_n[None, :]
        )

        a = tl.load(
            a_ptrs,
            mask=(offs_m[:, None] < M)
                 & ((k * BLOCK_K + offs_k[None, :]) < K),
            other=0
        )

        b = tl.load(
            b_ptrs,
            mask=((k * BLOCK_K + offs_k[:, None]) < K)
                 & (offs_n[None, :] < N),
            other=0
        )

        acc += tl.dot(a, b)

    c_ptrs = C + (
        offs_m[:, None] * N
        + offs_n[None, :]
    )

    tl.store(
        c_ptrs,
        acc.to(tl.float16),
        mask=(offs_m[:, None] < M)
             & (offs_n[None, :] < N)
    )

### Compile and Validate GEMM

In [8]:
M = N = K = 512

A = torch.randn(
    (M, K),
    device="cuda",
    dtype=torch.float16
)

B = torch.randn(
    (K, N),
    device="cuda",
    dtype=torch.float16
)

C = torch.empty(
    (M, N),
    device="cuda",
    dtype=torch.float16
)

compiled_kernel = gemm_kernel[
    (triton.cdiv(M, 32), triton.cdiv(N, 32))
](
    A, B, C,
    M, N, K,
    BLOCK_M=32,
    BLOCK_N=32,
    BLOCK_K=32,
    num_warps=4
)

torch.cuda.synchronize()

C_ref = A @ B

max_error = (C.float() - C_ref.float()).abs().max().item()

print("=" * 65)
print("GEMM Correctness")
print("=" * 65)

print("Shape:", (M, N, K))
print("Max absolute error:", max_error)

torch.testing.assert_close(
    C, C_ref,
    atol=0.1,
    rtol=0.01
)

print("GEMM Correctness: PASS")

GEMM Correctness
Shape: (512, 512, 512)
Max absolute error: 0.03125
GEMM Correctness: PASS


### Extract Compiler IR

In [9]:
print("=" * 65)
print("Triton Compiler IR Extraction")
print("=" * 65)

print("Available compilation stages:")
print(list(compiled_kernel.asm.keys()))

ir_names = ["ttir", "ttgir", "llir", "ptx"]

for name in ir_names:
    ir = compiled_kernel.asm.get(name)

    if ir is None:
        print(f"{name:8s}: NOT AVAILABLE")
        continue

    path = DIRS["llvm_ir"] / f"gemm_{name}.txt"

    path.write_text(str(ir))

    print(
        f"{name:8s}: "
        f"{len(str(ir)):8d} characters "
        f"-> {path}"
    )

Triton Compiler IR Extraction
Available compilation stages:
['source', 'ttir', 'ttgir', 'llir', 'ptx', 'cubin']
ttir    :     9335 characters -> /content/gemm_llvm_optimization/llvm_ir/gemm_ttir.txt
ttgir   :    10716 characters -> /content/gemm_llvm_optimization/llvm_ir/gemm_ttgir.txt
llir    :    48657 characters -> /content/gemm_llvm_optimization/llvm_ir/gemm_llir.txt
ptx     :    23735 characters -> /content/gemm_llvm_optimization/llvm_ir/gemm_ptx.txt


### LLVM IR Instruction Statistics

In [10]:
import re
from collections import Counter

llvm_ir = str(compiled_kernel.asm["llir"])

patterns = {
    "Basic Blocks": r"(?m)^[A-Za-z$._][\w$.-]*:",
    "PHI Nodes": r"\bphi\b",
    "Conditional / Unconditional Branch": r"\bbr\b",
    "GEP": r"\bgetelementptr\b",
    "Load": r"\bload\b",
    "Store": r"\bstore\b",
    "Integer Add": r"\badd\b",
    "Integer Multiply": r"\bmul\b",
    "Integer Compare": r"\bicmp\b",
}

print("=" * 65)
print("LLVM IR Structural Statistics")
print("=" * 65)

for name, pattern in patterns.items():
    count = len(re.findall(pattern, llvm_ir))
    print(f"{name:38s}: {count}")

LLVM IR Structural Statistics
Basic Blocks                          : 0
PHI Nodes                             : 2
Conditional / Unconditional Branch    : 2
GEP                                   : 91
Load                                  : 80
Store                                 : 8
Integer Add                           : 2
Integer Multiply                      : 0
Integer Compare                       : 6


In [11]:
import re

llvm_ir = str(compiled_kernel.asm["llir"])
lines = llvm_ir.splitlines()

# Match numeric, named, and quoted LLVM labels
label_pattern = re.compile(
    r'^\s*([A-Za-z$._][\w$.-]*|\d+|"[^"]+"):\s*(?:;.*)?$'
)

basic_blocks = []
phi_nodes = []
branches = []

for line_num, line in enumerate(lines, start=1):

    match = label_pattern.match(line)

    if match:
        basic_blocks.append((line_num, match.group(1)))

    # Match actual PHI instructions
    if re.search(r'^\s*(?:%[\w.$"-]+)\s*=\s*phi\b', line):
        phi_nodes.append((line_num, line.strip()))

    # Match branch instructions
    if re.match(r'^\s*br\s', line):
        branches.append((line_num, line.strip()))


print("=" * 70)
print("GEMM LLVM IR — CFG Components")
print("=" * 70)

print("\nBasic Blocks:")
for line_num, label in basic_blocks:
    print(f"Line {line_num:4d}: {label}")

print("\nPHI Nodes:")
for line_num, instruction in phi_nodes:
    print(f"Line {line_num:4d}: {instruction}")

print("\nBranch Instructions:")
for line_num, instruction in branches:
    print(f"Line {line_num:4d}: {instruction}")

print("\nSummary:")
print("Explicit Basic Block labels:", len(basic_blocks))
print("PHI Nodes:", len(phi_nodes))
print("Branches:", len(branches))

GEMM LLVM IR — CFG Components

Basic Blocks:
Line  115: 108
Line  419: 396

PHI Nodes:
Line  116: %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %108 ]
Line  117: %109 = phi <8 x float> [ zeroinitializer, %5 ], [ %395, %108 ]

Branch Instructions:
Line  113: br label %108, !dbg !17
Line  417: br i1 %exitcond.not, label %396, label %108, !dbg !17

Summary:
Explicit Basic Block labels: 2
PHI Nodes: 2
Branches: 2


### Inspect GEMM Loop Header and Latch

In [12]:
llvm_lines = llvm_ir.splitlines()

def show_ir_context(center_line, before=5, after=12):
    start = max(0, center_line - before - 1)
    end = min(len(llvm_lines), center_line + after)

    for i in range(start, end):
        print(f"{i+1:4d}: {llvm_lines[i]}")

print("=" * 70)
print("Loop Header — Block 108")
print("=" * 70)
show_ir_context(115, before=5, after=18)

print("\n" + "=" * 70)
print("Loop Latch — Branch at Line 417")
print("=" * 70)
show_ir_context(417, before=15, after=8)

print("\n" + "=" * 70)
print("Loop Exit — Block 396")
print("=" * 70)
show_ir_context(419, before=2, after=14)

Loop Header — Block 108
 110:   %105 = getelementptr inbounds nuw i8, ptr addrspace(3) %27, i32 2276
 111:   %106 = getelementptr inbounds nuw i8, ptr addrspace(3) %27, i32 2292
 112:   %107 = getelementptr inbounds nuw i8, ptr addrspace(3) %27, i32 2296
 113:   br label %108, !dbg !17
 114: 
 115: 108:                                              ; preds = %5, %108
 116:   %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %108 ]
 117:   %109 = phi <8 x float> [ zeroinitializer, %5 ], [ %395, %108 ]
 118:   %110 = shl nuw nsw i64 %indvars.iv, 5, !dbg !18
 119:   %gep = getelementptr half, ptr addrspace(1) %invariant.gep, i64 %110, !dbg !19
 120:   %111 = or disjoint i64 %110, %77, !dbg !20
 121:   %.idx = shl i64 %111, 10, !dbg !21
 122:   %gep211 = getelementptr i8, ptr addrspace(1) %invariant.gep210, i64 %.idx, !dbg !21
 123:   %112 = tail call { i32, i32, i32, i32 } asm sideeffect "mov.u32 $0, $4;\0A\09mov.u32 $1, $5;\0A\09mov.u32 $2, $6;\0A\09mov.u32 $3, $7;\0A\09@$9 ld.global.v

### Step 3.4 — Induction Variable Analysis

In [13]:
import re

llvm_ir = str(compiled_kernel.asm["llir"])

# Extract LLVM SSA instructions
instructions = {}

for line in llvm_ir.splitlines():
    match = re.match(
        r'^\s*(%[\w.$-]+)\s*=\s*(.+)$',
        line
    )
    if match:
        instructions[match.group(1)] = match.group(2).strip()


def show_instruction(name):
    instruction = instructions.get(name)

    if instruction:
        print(f"{name} = {instruction}")
    else:
        print(f"{name}: NOT FOUND")


print("=" * 70)
print("GEMM Induction Variable Analysis")
print("=" * 70)

targets = [
    "%indvars.iv",
    "%indvars.iv.next",
    "%110",
    "%gep",
    "%exitcond.not"
]

for name in targets:
    show_instruction(name)

GEMM Induction Variable Analysis
%indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %108 ]
%indvars.iv.next = add nuw nsw i64 %indvars.iv, 1, !dbg !17
%110 = shl nuw nsw i64 %indvars.iv, 5, !dbg !18
%gep = getelementptr half, ptr addrspace(1) %invariant.gep, i64 %110, !dbg !19
%exitcond.not = icmp eq i64 %indvars.iv.next, 16, !dbg !17


In [14]:
import pandas as pd

BLOCK_K = 32
ELEMENT_SIZE = 2  # FP16 = 2 bytes
NUM_ITERATIONS = 16

rows = []

for k in range(NUM_ITERATIONS):

    element_offset = k * BLOCK_K
    byte_offset = element_offset * ELEMENT_SIZE

    rows.append({
        "Iteration": k,
        "Induction Variable": k,
        "Element Offset": element_offset,
        "Byte Offset": byte_offset,
        "Byte Increment": BLOCK_K * ELEMENT_SIZE
    })

df = pd.DataFrame(rows)

display(df)

,Iteration,Induction Variable,Element Offset,Byte Offset,Byte Increment
0,0,0,0,0,64
1,1,1,32,64,64
2,2,2,64,128,64
3,3,3,96,192,64
4,4,4,128,256,64
5,5,5,160,320,64
6,6,6,192,384,64
7,7,7,224,448,64
8,8,8,256,512,64
9,9,9,288,576,64


In [15]:
import re

def extract_ssa_uses(instruction):
    return re.findall(r'%[\w.$-]+', instruction)


def trace_dependencies(name, depth=0, visited=None, max_depth=5):
    if visited is None:
        visited = set()

    indent = "  " * depth

    if name in visited:
        print(f"{indent}{name} (already visited)")
        return

    visited.add(name)

    instruction = instructions.get(name)

    if instruction is None:
        print(f"{indent}{name} (argument / external value)")
        return

    print(f"{indent}{name} = {instruction}")

    if depth >= max_depth:
        return

    for dependency in extract_ssa_uses(instruction):
        if dependency != name:
            trace_dependencies(
                dependency,
                depth + 1,
                visited,
                max_depth
            )


print("=" * 70)
print("GEMM Pointer Dependency Analysis")
print("=" * 70)

trace_dependencies("%gep")

GEMM Pointer Dependency Analysis
%gep = getelementptr half, ptr addrspace(1) %invariant.gep, i64 %110, !dbg !19
  %invariant.gep = getelementptr half, ptr addrspace(1) %0, i64 %76, !dbg !17
    %0 (argument / external value)
    %76 = sext i32 %20 to i64, !dbg !17
      %20 = or disjoint i32 %17, %13
        %17 = shl i32 %14, 9, !dbg !14
          %14 = or disjoint i32 %11, %8, !dbg !11
        %13 = shl nuw nsw i32 %12, 3, !dbg !10
          %12 = and i32 %9, 3, !dbg !10
  %110 = shl nuw nsw i64 %indvars.iv, 5, !dbg !18
    %indvars.iv = phi i64 [ 0, %5 ], [ %indvars.iv.next, %108 ]
      %5 (argument / external value)
      %indvars.iv.next = add nuw nsw i64 %indvars.iv, 1, !dbg !17
        %indvars.iv (already visited)
      %108 (argument / external value)


### Step 3.4 — LLVM's built-in Strength Reduction optimization.

### Create a Controlled LLVM IR Example

In [16]:
from pathlib import Path

ir_before = r"""
define void @gemm_address_loop(
    ptr %A,
    ptr %out,
    i64 %K
) {
entry:
  br label %loop

loop:
  %k = phi i64 [ 0, %entry ], [ %k.next, %loop ]

  %offset = shl i64 %k, 5
  %ptr = getelementptr half, ptr %A, i64 %offset

  %value = load half, ptr %ptr
  %out.ptr = getelementptr half, ptr %out, i64 %k
  store volatile half %value, ptr %out.ptr

  %k.next = add nuw i64 %k, 1
  %done = icmp eq i64 %k.next, 16
  br i1 %done, label %exit, label %loop

exit:
  ret void
}
"""

input_path = DIRS["llvm_ir"] / "address_before.ll"
input_path.write_text(ir_before)

print(ir_before)
print("\nSaved:", input_path)


define void @gemm_address_loop(
    ptr %A,
    ptr %out,
    i64 %K
) {
entry:
  br label %loop

loop:
  %k = phi i64 [ 0, %entry ], [ %k.next, %loop ]

  %offset = shl i64 %k, 5
  %ptr = getelementptr half, ptr %A, i64 %offset

  %value = load half, ptr %ptr
  %out.ptr = getelementptr half, ptr %out, i64 %k
  store volatile half %value, ptr %out.ptr

  %k.next = add nuw i64 %k, 1
  %done = icmp eq i64 %k.next, 16
  br i1 %done, label %exit, label %loop

exit:
  ret void
}


Saved: /content/gemm_llvm_optimization/llvm_ir/address_before.ll


### Run LLVM Loop Strength Reduction

In [17]:
import subprocess

output_path = DIRS["optimized_ir"] / "address_after.ll"

cmd = [
    "opt-20",
    "-passes=loop-simplify,loop-reduce",
    "-verify-each",
    "-S",
    str(input_path),
    "-o",
    str(output_path)
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("Return code:", result.returncode)

if result.returncode != 0:
    print("LLVM error:")
    print(result.stderr)
else:
    print("LLVM optimization: PASS")
    print("\n=== Optimized LLVM IR ===")
    print(output_path.read_text())

Return code: 0
LLVM optimization: PASS

=== Optimized LLVM IR ===
; ModuleID = '/content/gemm_llvm_optimization/llvm_ir/address_before.ll'
source_filename = "/content/gemm_llvm_optimization/llvm_ir/address_before.ll"

define void @gemm_address_loop(ptr %A, ptr %out, i64 %K) {
entry:
  br label %loop

loop:                                             ; preds = %loop, %entry
  %k = phi i64 [ 0, %entry ], [ %k.next, %loop ]
  %offset = shl i64 %k, 5
  %ptr = getelementptr half, ptr %A, i64 %offset
  %value = load half, ptr %ptr, align 2
  %out.ptr = getelementptr half, ptr %out, i64 %k
  store volatile half %value, ptr %out.ptr, align 2
  %k.next = add nuw i64 %k, 1
  %done = icmp eq i64 %k.next, 16
  br i1 %done, label %exit, label %loop

exit:                                             ; preds = %loop
  ret void
}



### Compare Before and After

In [18]:
import difflib

before = input_path.read_text()

if output_path.exists() and result.returncode == 0:
    after = output_path.read_text()

    diff = difflib.unified_diff(
        before.splitlines(),
        after.splitlines(),
        fromfile="Before",
        tofile="After",
        lineterm=""
    )

    print("\n".join(diff))
else:
    print("Optimization output unavailable")

--- Before
+++ After
@@ -1,26 +1,21 @@
+; ModuleID = '/content/gemm_llvm_optimization/llvm_ir/address_before.ll'
+source_filename = "/content/gemm_llvm_optimization/llvm_ir/address_before.ll"
 
-define void @gemm_address_loop(
-    ptr %A,
-    ptr %out,
-    i64 %K
-) {
+define void @gemm_address_loop(ptr %A, ptr %out, i64 %K) {
 entry:
   br label %loop
 
-loop:
+loop:                                             ; preds = %loop, %entry
   %k = phi i64 [ 0, %entry ], [ %k.next, %loop ]
-
   %offset = shl i64 %k, 5
   %ptr = getelementptr half, ptr %A, i64 %offset
-
-  %value = load half, ptr %ptr
+  %value = load half, ptr %ptr, align 2
   %out.ptr = getelementptr half, ptr %out, i64 %k
-  store volatile half %value, ptr %out.ptr
-
+  store volatile half %value, ptr %out.ptr, align 2
   %k.next = add nuw i64 %k, 1
   %done = icmp eq i64 %k.next, 16
   br i1 %done, label %exit, label %loop
 
-exit:
+exit:                                             ; preds = %loop
   ret void
 }


### Test Loop Strength Reduction with Target Information

In [19]:
import subprocess
from pathlib import Path

input_file = DIRS["llvm_ir"] / "address_before.ll"
output_file = DIRS["optimized_ir"] / "address_x86_optimized.ll"

cmd = [
    "opt-20",
    "-mtriple=x86_64-unknown-linux-gnu",
    "-passes=loop-simplify,indvars,loop-reduce",
    "-verify-each",
    "-S",
    str(input_file),
    "-o",
    str(output_file)
]

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True
)

print("=" * 70)
print("LLVM Loop Strength Reduction — x86_64")
print("=" * 70)

print("Return code:", result.returncode)

if result.returncode == 0:
    optimized_ir = output_file.read_text()
    print(optimized_ir)
else:
    print(result.stderr)

LLVM Loop Strength Reduction — x86_64
Return code: 0
; ModuleID = '/content/gemm_llvm_optimization/llvm_ir/address_before.ll'
source_filename = "/content/gemm_llvm_optimization/llvm_ir/address_before.ll"
target datalayout = "e-m:e-p270:32:32-p271:32:32-p272:64:64-i64:64-i128:128-f80:128-n8:16:32:64-S128"
target triple = "x86_64-unknown-linux-gnu"

define void @gemm_address_loop(ptr %A, ptr %out, i64 %K) {
entry:
  br label %loop

loop:                                             ; preds = %loop, %entry
  %lsr.iv2 = phi ptr [ %scevgep3, %loop ], [ %A, %entry ]
  %lsr.iv = phi i64 [ %lsr.iv.next, %loop ], [ -32, %entry ]
  %value = load half, ptr %lsr.iv2, align 2
  %scevgep = getelementptr i8, ptr %out, i64 %lsr.iv
  %scevgep1 = getelementptr i8, ptr %scevgep, i64 32
  store volatile half %value, ptr %scevgep1, align 2
  %lsr.iv.next = add nsw i64 %lsr.iv, 2
  %scevgep3 = getelementptr i8, ptr %lsr.iv2, i64 64
  %done = icmp eq i64 %lsr.iv.next, 0
  br i1 %done, label %exit, label %loop

### Automatically Detect Strength Reduction

In [20]:
import re

if result.returncode == 0:

    optimized_ir = output_file.read_text()

    checks = {
        "Original shift exists":
            bool(re.search(r'%offset\s*=\s*shl\b', optimized_ir)),

        "Original address GEP exists":
            bool(re.search(r'%ptr\s*=\s*getelementptr\b', optimized_ir)),

        "Pointer PHI exists":
            bool(re.search(r'=\s*phi\s+ptr\b', optimized_ir)),

        "Original integer PHI exists":
            bool(re.search(r'%k\s*=\s*phi\b', optimized_ir)),
    }

    print("=" * 70)
    print("Strength Reduction Analysis")
    print("=" * 70)

    for name, value in checks.items():
        print(f"{name:35s}: {value}")

    print("\nNote: These are structural checks, not proof of correctness.")

Strength Reduction Analysis
Original shift exists              : False
Original address GEP exists        : False
Pointer PHI exists                 : True
Original integer PHI exists        : False

Note: These are structural checks, not proof of correctness.


### 证明经过 LSR 优化的 LLVM IR 与原始 LLVM IR 产生相同结果

In [21]:
import subprocess
import ctypes
import numpy as np

before_ll = DIRS["llvm_ir"] / "address_before.ll"
after_ll = DIRS["optimized_ir"] / "address_x86_optimized.ll"

before_so = DIRS["results"] / "before.so"
after_so = DIRS["results"] / "after.so"

for source, output in [
    (before_ll, before_so),
    (after_ll, after_so),
]:
    subprocess.run(
        [
            "clang-20",
            "-shared",
            "-fPIC",
            "-O0",
            str(source),
            "-o",
            str(output)
        ],
        check=True
    )

print("Both LLVM IR versions compiled successfully.")

Both LLVM IR versions compiled successfully.


In [22]:
def load_kernel(path):
    lib = ctypes.CDLL(str(path))
    fn = lib.gemm_address_loop

    fn.argtypes = [
        ctypes.c_void_p,
        ctypes.c_void_p,
        ctypes.c_int64
    ]
    fn.restype = None

    return lib, fn


lib_before, fn_before = load_kernel(before_so)
lib_after, fn_after = load_kernel(after_so)

# 16 iterations, each reads A[k * 32]
A = np.arange(512, dtype=np.uint16)

out_before = np.zeros(16, dtype=np.uint16)
out_after = np.zeros(16, dtype=np.uint16)

fn_before(
    A.ctypes.data,
    out_before.ctypes.data,
    512
)

fn_after(
    A.ctypes.data,
    out_after.ctypes.data,
    512
)

expected = A[::32]

print("=" * 65)
print("LLVM Strength Reduction — Correctness Validation")
print("=" * 65)

print("Before  :", out_before)
print("After   :", out_after)
print("Expected:", expected)

np.testing.assert_array_equal(out_before, expected)
np.testing.assert_array_equal(out_after, expected)

print("\nCorrectness: PASS")

LLVM Strength Reduction — Correctness Validation
Before  : [  0  32  64  96 128 160 192 224 256 288 320 352 384 416 448 480]
After   : [  0  32  64  96 128 160 192 224 256 288 320 352 384 416 448 480]
Expected: [  0  32  64  96 128 160 192 224 256 288 320 352 384 416 448 480]

Correctness: PASS


### LLVM Pass Ablation Study

In [23]:
import subprocess
import re
import pandas as pd

experiments = [
    ("A", False, "loop-simplify,loop-reduce"),
    ("B", False, "loop-simplify,indvars,loop-reduce"),
    ("C", True,  "loop-simplify,loop-reduce"),
    ("D", True,  "loop-simplify,indvars,loop-reduce"),
]

results = []

for name, use_target, passes in experiments:

    output_file = DIRS["optimized_ir"] / f"ablation_{name}.ll"

    cmd = ["opt-20"]

    if use_target:
        cmd.append("-mtriple=x86_64-unknown-linux-gnu")

    cmd.extend([
        f"-passes={passes}",
        "-verify-each",
        "-S",
        str(input_path),
        "-o",
        str(output_file)
    ])

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True
    )

    if result.returncode != 0:
        print(f"Experiment {name} failed:")
        print(result.stderr)
        continue

    ir = output_file.read_text()

    results.append({
        "Experiment": name,
        "Target": "x86_64" if use_target else "Default",
        "Indvars Pass": "indvars" in passes,
        "Pointer PHI": bool(re.search(r"=\s*phi\s+ptr\b", ir)),
        "Original Shift": bool(re.search(r"%offset\s*=\s*shl\b", ir)),
        "Original k PHI": bool(re.search(r"%k\s*=\s*phi\b", ir)),
    })

df_ablation = pd.DataFrame(results)

display(df_ablation)

,Experiment,Target,Indvars Pass,Pointer PHI,Original Shift,Original k PHI
0,A,Default,False,False,True,True
1,B,Default,True,False,True,True
2,C,x86_64,False,True,False,False
3,D,x86_64,True,True,False,False


### Real Triton GEMM Backend Analysis

In [24]:
import re
from collections import Counter

ptx = str(compiled_kernel.asm["ptx"])

instruction_counts = Counter()

for line in ptx.splitlines():

    line = line.strip()

    # Remove optional PTX predicate
    line = re.sub(
        r"^@!?%[\w.]+\s+",
        "",
        line
    )

    match = re.match(
        r"^([A-Za-z][\w.]*)\s",
        line
    )

    if match:
        opcode = match.group(1).split(".")[0]
        instruction_counts[opcode] += 1


categories = {
    "Address Arithmetic": [
        "add", "sub", "mul", "mad", "shl", "shr",
        "cvta", "cvt", "mov"
    ],
    "Memory": [
        "ld", "st"
    ],
    "Control Flow": [
        "bra", "setp", "selp", "ret"
    ],
}

print("=" * 70)
print("GEMM PTX — Instruction Statistics")
print("=" * 70)

for category, opcodes in categories.items():

    print(f"\n{category}")
    print("-" * 50)

    for opcode in opcodes:
        print(
            f"{opcode:12s}: "
            f"{instruction_counts.get(opcode, 0)}"
        )

GEMM PTX — Instruction Statistics

Address Arithmetic
--------------------------------------------------
add         : 14
sub         : 0
mul         : 1
mad         : 3
shl         : 13
shr         : 0
cvta        : 0
cvt         : 24
mov         : 34

Memory
--------------------------------------------------
ld          : 121
st          : 9

Control Flow
--------------------------------------------------
bra         : 1
setp        : 3
selp        : 0
ret         : 0


In [25]:
patterns = [
    r"\bshl\.",
    r"\bmul\.",
    r"\bmul\.wide",
    r"\bmad\.",
    r"\badd\.s64",
    r"\badd\.u64",
    r"\bld\.global",
    r"\bbra\b",
]

for pattern in patterns:

    print("\n" + "=" * 70)
    print("Pattern:", pattern)
    print("=" * 70)

    matches = [
        line.strip()
        for line in ptx.splitlines()
        if re.search(pattern, line)
    ]

    print("Total matches:", len(matches))

    for line in matches[:8]:
        print(line)


Pattern: \bshl\.
Total matches: 13
shl.b32 	%r12, %r11, 5;
shl.b32 	%r14, %r3, 3;
shl.b32 	%r15, %r1, 5;
shl.b32 	%r6, %r4, 9;
shl.b32 	%r16, %r2, 5;
shl.b32 	%r19, %r2, 4;
shl.b32 	%r9, %r2, 3;
shl.b32 	%r22, %r11, 14;

Pattern: \bmul\.
Total matches: 1
mul.wide.u32 	%rd5, %r5, 2;

Pattern: \bmul\.wide
Total matches: 1
mul.wide.u32 	%rd5, %r5, 2;

Pattern: \bmad\.
Total matches: 3
mad.wide.u32 	%rd6, %r13, 1024, %rd5;
mad.wide.s32 	%rd1, %r25, 2, %rd3;
mad.wide.s32 	%rd8, %r495, 2, %rd2;

Pattern: \badd\.s64
Total matches: 4
add.s64 	%rd9, %rd4, %rd6;
add.s64 	%rd7, %rd1, %rd10;
add.s64 	%rd10, %rd10, 64;
add.s64 	%rd9, %rd9, 32768;

Pattern: \badd\.u64
Total matches: 0

Pattern: \bld\.global
Total matches: 2
@%p1 ld.global.v4.b32 { %r26, %r27, %r28, %r29 }, [ %rd7 + 0 ];
@%p2 ld.global.v4.b32 { %r31, %r32, %r33, %r34 }, [ %rd9 + 0 ];

Pattern: \bbra\b
Total matches: 1
@%p3 bra 	$L__BB0_1;
